# Lecture 11 — Edge AI: putting the model on the ESP32

**01211271 Industrial AI and IoT** · Electromechanical Manufacturing Engineering

---

For three lectures the models have lived on a laptop with NumPy, scikit-learn and as
much memory as they wanted. Today they move onto a microcontroller with 200 KB of
usable RAM, no NumPy, no hardware divider worth the name, and a hard deadline: the
sensor produces a new window every **128 ms**, whether or not you are finished with
the last one.

Nothing about the models changes. What changes is that every choice now has a price
you can measure in microseconds and bytes, and the measurement is the lecture.

The two artefacts we deploy are the ones you already built:

* `lecture9_model.json` — the linear SVM that names the fault.
* `lecture10_alarm.json` — the anomaly baseline that needs no labels.

They go on the device together, and by the end of this notebook you will see exactly
why neither one is enough on its own.

## Learning objectives

By the end of this notebook you should be able to:

1. Choose between edge, cloud and hybrid architectures on latency, bandwidth and what
   happens when the network is down — and defend the choice.
2. State a compute budget in microseconds per window and test a pipeline against it.
3. Write the twelve features in plain Python, including a radix-2 FFT, and verify them
   against the NumPy reference.
4. Decide whether the FFT earns its place, using measured accuracy and measured cost.
5. Export a linear model by folding the standardiser into the coefficients, and prove
   the fold is exact.
6. Say what single precision costs you, in relative error, feature by feature.
7. Design the fail-safe behaviour: what the device does when a feature, a model, or the
   network fails.
8. Explain why a confident classifier and an alarm must both run, and which one owns
   the actuator.

## Prerequisites

Lecture 8 (windowing and the twelve features), Lecture 9 (the linear SVM and its
exported JSON), Lecture 10 (the anomaly baseline, its threshold and its persistence
rule). Lab 8's timing measurement on Wokwi is used in section 3.

## The running example

The same rotor rig. Same twelve features, same three classes — plus, in section 8, a
fourth machine state that appears in no training set anywhere in this course.

In [1]:
import sys
import json
import math
import time
import subprocess

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn

print("python     ", sys.version.split()[0])
for m in (np, pd, matplotlib, sklearn):
    print(f"{m.__name__:<11}", m.__version__)

plt.rcParams.update({"figure.figsize": (10.5, 3.6), "font.size": 10,
                     "axes.grid": True, "grid.alpha": 0.35,
                     "axes.spines.top": False, "axes.spines.right": False})
BLUE, ORANGE, AQUA, VIOLET, RED = ("#2a78d6", "#eb6834", "#1baf7a",
                                   "#4a3aa7", "#e34948")
LAB = ["normal", "imbalance", "bearing"]

python      3.11.15
numpy       2.4.4
pandas      3.0.2
matplotlib  3.10.9
sklearn     1.8.0


## 1. Edge, cloud, hybrid

Three architectures are available to you, and the course has quietly been building
towards the third one.

| | everything on the device | everything in the cloud | hybrid |
|---|---|---|---|
| **latency to a decision** | one window, 128 ms | 128 ms + a round trip to the broker | local decision, cloud second opinion |
| **bandwidth** | 12 numbers per window | every sample, 2000 per second | features up, not waveforms |
| **if the network drops** | keeps deciding | blind | keeps deciding, reports later |
| **model you can run** | small and fixed | anything you like | small local, large remote |
| **updating the model** | reflash or OTA | change it whenever you like | local model updated from the cloud |

The bandwidth row is worth doing as arithmetic rather than taking on trust.

In [2]:
FS_HZ, WIN_N, HOP_N = 2000, 256, 128
BYTES_PER_SAMPLE = 2                      # int16 counts from the ADC

raw_bps = FS_HZ * BYTES_PER_SAMPLE
windows_per_s = FS_HZ / HOP_N
feat_bps = windows_per_s * 12 * 4         # twelve float32 per window

print(f"raw waveform          {raw_bps:8.0f} bytes/s")
print(f"twelve features       {feat_bps:8.0f} bytes/s")
print(f"ratio                 {raw_bps / feat_bps:8.1f} x less traffic")
print()
print(f"one window is {1000 * WIN_N / FS_HZ:.0f} ms; a new one starts every "
      f"{1000 * HOP_N / FS_HZ:.0f} ms")

raw waveform              4000 bytes/s
twelve features            750 bytes/s
ratio                      5.3 x less traffic

one window is 128 ms; a new one starts every 64 ms


Five times less traffic, and the reduction is *free* — the features were computed on
the device anyway, because the device needs them to decide. Five is the conservative
figure, too: it assumes you publish every window. Lecture 12 will publish on change
and on alarm, and the ratio becomes hundreds to one.

The last line is the number this whole lecture is organised around. A window is 128 ms
long and a new one begins every 64 ms; we will budget against the **window duration**,
128 ms, because that is the interval the device must not exceed on average if it is to
keep up with its own sensor. Fall behind and the buffer grows without limit, which on a
device with 200 KB of RAM is a crash with a delay fuse in it.

## 2. The twelve features, in plain Python

Lab 8 wrote the seven time-domain features on the device. The five frequency-domain
ones need a spectrum, and there is no `np.fft` on a microcontroller, so we write one.

Here is the time-domain half as it ships in `features.py`. Read it for what it does
*not* do: it allocates nothing, it makes two passes over the buffer rather than seven,
and it never builds an intermediate list.

In [3]:
def time_features(buf):
    """Seven time-domain features.  Two passes, no allocation.

    Returns (mean, rms, std, ptp, crest, kurt, zcr).
    """
    n = len(buf)
    total = 0.0
    lo = hi = buf[0]
    for v in buf:
        total += v
        if v < lo:
            lo = v
        if v > hi:
            hi = v
    mean = total / n
    ptp = hi - lo

    s2 = 0.0
    s4 = 0.0
    peak = 0.0
    zc = 0
    prev = buf[0] - mean
    for v in buf:
        d = v - mean
        dd = d * d
        s2 += dd
        s4 += dd * dd
        if dd > peak:
            peak = dd
        if (d < 0.0) != (prev < 0.0):
            zc += 1
        prev = d

    rms = (s2 / n) ** 0.5
    r2 = rms * rms
    crest = (peak ** 0.5) / rms if rms > 0.0 else 0.0
    kurt = (s4 / n) / (r2 * r2) if rms > 0.0 else 0.0
    return (mean, rms, rms, ptp, crest, kurt, zc / (n - 1))

Two passes, not seven, because each pass over 256 floats in MicroPython is roughly
16 µs of interpreter overhead before any arithmetic happens. On a laptop you write
`x.mean()`, `x.std()`, `np.ptp(x)` and think nothing of it; here each of those is a
separate walk down the same list.

Now the FFT — radix-2 decimation-in-time, in-place, with the twiddle factors and the
bit-reversal permutation computed **once at import** and never again.

In [4]:
WIN, FS = 256, 2000.0
DF = FS / WIN                     # 7.8125 Hz per bin

BANDS = ((20.0, 42.0),            # e_1x    shaft rate
         (48.0, 78.0),            # e_2x
         (90.0, 132.0),           # e_bpfo  outer-race defect order
         (400.0, 900.0))          # e_hi    housing resonance

from array import array

_han = array('f', [0.5 - 0.5 * math.cos(2.0 * math.pi * i / (WIN - 1))
                   for i in range(WIN)])
_re = array('f', [0.0] * WIN)
_im = array('f', [0.0] * WIN)
_cos = array('f', [math.cos(-2.0 * math.pi * k / WIN) for k in range(WIN // 2)])
_sin = array('f', [math.sin(-2.0 * math.pi * k / WIN) for k in range(WIN // 2)])


def _bitrev_table(n):
    bits = 0
    while (1 << bits) < n:
        bits += 1
    t = array('H', [0] * n)
    for i in range(n):
        r, x = 0, i
        for _ in range(bits):
            r = (r << 1) | (x & 1)
            x >>= 1
        t[i] = r
    return t


_rev = _bitrev_table(WIN)
_BAND_BINS = tuple((max(1, int(lo / DF + 0.9999)), int(hi / DF)) for lo, hi in BANDS)

buffer_bytes = 4 * WIN * 3 + 4 * (WIN // 2) * 2 + 2 * WIN
print(f"work buffers allocated once at import: {buffer_bytes} bytes")
print(f"frequency resolution: {DF:.4f} Hz per bin")
print(f"band edges as bin indices: {_BAND_BINS}")

work buffers allocated once at import: 4608 bytes
frequency resolution: 7.8125 Hz per bin
band edges as bin indices: ((3, 5), (7, 9), (12, 16), (52, 115))


In [5]:
def _fft(buf, mean):
    """In-place radix-2 decimation-in-time FFT of (buf - mean) * hanning.

    Writes into the module-level _re and _im.  n must be a power of two.
    """
    n = WIN
    rev = _rev
    re = _re
    im = _im
    han = _han
    for i in range(n):                       # load, de-mean, window, bit-reverse
        re[rev[i]] = (buf[i] - mean) * han[i]
        im[i] = 0.0

    size = 2
    while size <= n:
        half = size >> 1
        step = n // size
        for start in range(0, n, size):
            k = 0
            for j in range(start, start + half):
                wr = _cos[k]
                wi = _sin[k]
                j2 = j + half
                tr = wr * re[j2] - wi * im[j2]
                ti = wr * im[j2] + wi * re[j2]
                re[j2] = re[j] - tr
                im[j2] = im[j] - ti
                re[j] += tr
                im[j] += ti
                k += step
        size <<= 1
    return re, im

In [6]:
def freq_features(buf, mean):
    """Dominant frequency plus four normalised band energies.

    Returns (dom_freq, e_1x, e_2x, e_bpfo, e_hi).
    """
    re, im = _fft(buf, mean)

    half = WIN // 2
    total = 0.0
    best = 0.0
    best_k = 0
    # bin 0 is the DC term; we removed the mean, so it carries no information
    for k in range(1, half + 1):
        p = re[k] * re[k] + im[k] * im[k]
        total += p
        if p > best:
            best = p
            best_k = k

    if total <= 0.0:
        return (0.0, 0.0, 0.0, 0.0, 0.0)

    out = [best_k * DF]
    for k0, k1 in _BAND_BINS:
        s = 0.0
        for k in range(k0, k1 + 1):
            s += re[k] * re[k] + im[k] * im[k]
        out.append(s / total)
    return tuple(out)


def all_features(buf):
    """The twelve features, in the order the model expects."""
    t = time_features(buf)
    f = freq_features(buf, t[0])
    return (t[0], t[1], t[2], t[3], t[4], t[5], t[6],
            f[0], f[1], f[2], f[3], f[4])

In [7]:
# Keep handles to the device versions under their own names. Below we run rig.py,
# which defines NumPy functions called `time_features` and `freq_features` that take
# a whole matrix of windows at once -- same names, completely different functions.
# A silent collision like this is the most common way a "verified" edge port turns
# out to have verified nothing.
time_features_py = time_features
freq_features_py = freq_features

print("device time_features on a 4-sample square wave:")
print("  ", time_features_py([0.0, 1.0, 0.0, -1.0]))

device time_features on a 4-sample square wave:
   (0.0, 0.7071067811865476, 0.7071067811865476, 2.0, 1.414213562373095, 1.9999999999999991, 0.3333333333333333)


Three details in there are the difference between code that runs for a year and code
that does not:

* `_re`, `_im`, `_han`, `_cos`, `_sin`, `_rev` are **module-level and reused**. Allocating
  them inside `_fft` would put 4.5 KB of garbage on the heap every 64 ms, fragment it,
  and fail at three in the morning after a fortnight.
* They are `array('f')`, not lists. A MicroPython list of 256 floats is 256 boxed float
  objects plus a pointer array — several kilobytes and a lot of pointer chasing. An
  `array('f')` is 1024 bytes of contiguous single precision.
* The twiddle factors are precomputed. `math.cos` inside the butterfly loop would be
  called 1024 times per window.

### Verify before you optimise

The rule from Lab 8 applies with more force here: device code that is fast and wrong is
worse than laptop code that is slow and right. Check the device implementation against
the NumPy reference on real data before trusting a single timing number.

In [8]:
"""
rig.py — the rotor-rig simulator shared by Lectures 8-14
01211271 Industrial AI and IoT

Synthetic but physics-motivated vibration data for a small motor-driven rotor rig.

Rig model
---------
A 3-phase induction motor drives a rotor disc supported by two rolling-element
bearings.  A MEMS accelerometer is mounted radially on the drive-end bearing
housing and sampled at fs = 2000 Hz.

Three machine states are simulated:

  normal    : residual unbalance only -> modest 1x shaft-rate component
  imbalance : added trial mass on the disc -> large 1x component
  bearing   : outer-race spall -> periodic impulses at BPFO that ring the
              bearing housing resonance

Outer-race defect frequency, for a bearing with n = 9 balls, ball/pitch
diameter ratio d/D = 0.2 and contact angle 0:

    BPFO = (n/2) * (1 - (d/D) cos(phi)) * f_r = 4.5 * 0.8 * f_r = 3.6 * f_r

so at a nominal 30 Hz shaft rate (1800 rpm) the impulses arrive at 108 Hz.

Every run carries its own nuisance variation -- shaft speed, sensor gain,
broadband noise floor, DC bias and a structural tone.  This is what makes a
random train/test split across overlapping windows dishonest: the model can
memorise the run, not the fault.

Lecture 8 built the classification dataset from make_dataset().  Lecture 9 adds
make_speed_sweep(), a separate acquisition campaign used for the soft-sensor
thread: the rig is run across its speed range in the healthy state so that a
regression model can learn to read shaft speed off the vibration alone.
"""

import numpy as np

# ----------------------------------------------------------------- constants
FS = 2000.0                 # sampling rate, Hz
RUN_SECONDS = 4.0           # length of one acquisition run
RUNS_PER_CLASS = 16
CLASSES = ("normal", "imbalance", "bearing")

WIN = 256                   # default window length, samples (128 ms)
HOP = 128                   # 50 % overlap

F_RESONANCE = 600.0         # bearing housing resonance, Hz
TAU_RING = 0.0012           # impulse ring-down time constant, s
BPFO_RATIO = 3.6            # outer-race defect order

# Fixed analysis bands, Hz.  Shaft speed wanders over 28-32 Hz, so the bands
# are wide enough to hold the order they are named for without a tachometer.
BANDS = {
    "e_1x":   (20.0, 42.0),     # shaft rate
    "e_2x":   (48.0, 78.0),     # twice shaft rate
    "e_bpfo": (90.0, 132.0),    # outer-race defect order
    "e_hi":   (400.0, 900.0),   # housing resonance region
}

FEATURE_NAMES = [
    "mean", "rms", "std", "ptp", "crest", "kurt", "zcr",
    "dom_freq", "e_1x", "e_2x", "e_bpfo", "e_hi",
]


# ------------------------------------------------------------ signal synthesis
def _run_params(rng, state):
    """Nuisance parameters that vary from one acquisition run to the next."""
    return dict(
        f_r=rng.uniform(28.0, 32.0),          # shaft rate, Hz
        gain=rng.uniform(0.85, 1.25),         # sensor / mounting gain
        noise=rng.uniform(0.04, 0.20),        # broadband floor, g rms
        bias=rng.uniform(-0.06, 0.06),        # accelerometer DC offset, g
        f_struct=rng.uniform(150.0, 700.0),   # frame resonance -- a confounder
        a_struct=rng.uniform(0.03, 0.15),
        a_1x=(rng.uniform(0.20, 0.38) if state != "imbalance"
              else rng.uniform(0.58, 1.10)),
        a_imp=(rng.uniform(0.45, 1.40) if state == "bearing" else 0.0),
    )


def _synth(rng, p, seconds=RUN_SECONDS, fs=FS):
    """Turn a parameter dict into a waveform.  Shared by every campaign."""
    n = int(seconds * fs)
    t = np.arange(n) / fs

    ph = rng.uniform(0, 2 * np.pi, 4)
    x = (p["a_1x"] * np.sin(2 * np.pi * p["f_r"] * t + ph[0])
         + 0.32 * p["a_1x"] * np.sin(2 * np.pi * 2 * p["f_r"] * t + ph[1])
         + 0.12 * p["a_1x"] * np.sin(2 * np.pi * 3 * p["f_r"] * t + ph[2])
         + p["a_struct"] * np.sin(2 * np.pi * p["f_struct"] * t + ph[3]))

    if p["a_imp"] > 0:
        f_bpfo = BPFO_RATIO * p["f_r"]
        period = fs / f_bpfo
        k = 0
        while True:
            # 1 % random slip, as real rolling elements do
            idx = int(k * period * (1.0 + rng.normal(0, 0.01)))
            if idx >= n:
                break
            tail = np.arange(n - idx) / fs
            ring = (p["a_imp"] * rng.uniform(0.75, 1.25)
                    * np.exp(-tail / TAU_RING)
                    * np.sin(2 * np.pi * F_RESONANCE * tail))
            x[idx:] += ring
            k += 1

    x = p["gain"] * (x + rng.normal(0, p["noise"], n)) + p["bias"]
    return x.astype(np.float64)


def make_run(rng, state, seconds=RUN_SECONDS, fs=FS):
    """Synthesise one acquisition run.  Returns (signal, params)."""
    p = _run_params(rng, state)
    return _synth(rng, p, seconds, fs), p


def make_dataset(seed=7, runs_per_class=RUNS_PER_CLASS):
    """All runs.  Returns list of dicts with signal, label and run id."""
    rng = np.random.default_rng(seed)
    runs, rid = [], 0
    for state in CLASSES:
        for _ in range(runs_per_class):
            x, p = make_run(rng, state)
            runs.append({"x": x, "label": state, "run": rid, "params": p})
            rid += 1
    return runs


# ------------------------------------------------------------------- features
def frame(x, win=WIN, hop=HOP):
    """Slice a 1-D signal into overlapping windows -> (n_windows, win)."""
    n = 1 + (len(x) - win) // hop
    idx = np.arange(win)[None, :] + hop * np.arange(n)[:, None]
    return x[idx]


def time_features(w):
    """Seven time-domain features for each row of w."""
    mean = w.mean(axis=1)
    ac = w - mean[:, None]                      # remove DC before anything else
    rms = np.sqrt((ac ** 2).mean(axis=1))
    std = ac.std(axis=1)
    ptp = np.ptp(w, axis=1)
    peak = np.abs(ac).max(axis=1)
    crest = peak / np.maximum(rms, 1e-12)
    m4 = (ac ** 4).mean(axis=1)
    kurt = m4 / np.maximum(rms, 1e-12) ** 4     # non-excess kurtosis; 3.0 = Gaussian
    zc = np.diff(np.signbit(ac).astype(np.int8), axis=1)
    zcr = np.abs(zc).sum(axis=1) / (w.shape[1] - 1)
    return np.column_stack([mean, rms, std, ptp, crest, kurt, zcr])


def freq_features(w, fs=FS, normalise=True):
    """Dominant frequency plus four band energies.

    normalise=True gives each band as a fraction of the window's total power,
    which makes the feature independent of sensor gain.  normalise=False gives
    the absolute band power, whose units and magnitude differ wildly from the
    time-domain features -- useful for showing when feature scaling matters.
    """
    n = w.shape[1]
    win = np.hanning(n)
    ac = w - w.mean(axis=1, keepdims=True)
    spec = np.abs(np.fft.rfft(ac * win, axis=1)) ** 2
    f = np.fft.rfftfreq(n, 1.0 / fs)
    spec[:, 0] = 0.0                            # DC carries no information here
    total = np.maximum(spec.sum(axis=1), 1e-20)

    dom = f[spec.argmax(axis=1)]
    cols = [dom]
    for lo, hi in BANDS.values():
        m = (f >= lo) & (f < hi)
        band = spec[:, m].sum(axis=1)
        cols.append(band / total if normalise else band)
    return np.column_stack(cols)


def features_of(w, fs=FS, normalise=True):
    """Full 12-feature vector for each row of w."""
    return np.column_stack([time_features(w), freq_features(w, fs, normalise)])


def build_table(runs, win=WIN, hop=HOP, fs=FS, normalise=True):
    """Turn the raw runs into the (X, y, groups) learning problem."""
    X, y, g = [], [], []
    for r in runs:
        w = frame(r["x"], win, hop)
        X.append(features_of(w, fs, normalise))
        y += [r["label"]] * w.shape[0]
        g += [r["run"]] * w.shape[0]
    return np.vstack(X), np.array(y), np.array(g)


def raw_table(runs, win=WIN, hop=HOP):
    """The naive alternative: the raw samples of each window as the features."""
    X, y, g = [], [], []
    for r in runs:
        w = frame(r["x"], win, hop)
        X.append(w)
        y += [r["label"]] * w.shape[0]
        g += [r["run"]] * w.shape[0]
    return np.vstack(X), np.array(y), np.array(g)


# ------------------------------------------------- Lecture 9: the speed sweep
SWEEP_RUNS = 40
SWEEP_FR = (22.0, 40.0)     # stays inside the e_1x band (20-42 Hz)


def make_speed_sweep(seed=11, n_runs=SWEEP_RUNS, seconds=RUN_SECONDS):
    """A healthy-machine speed sweep, for the virtual tachometer.

    The rig has no tachometer.  To build one in software you run the machine
    across its speed range in a known-good state and record the true shaft
    speed from the drive's own setpoint.  Each run still carries its own gain,
    noise floor, bias and frame resonance -- the soft sensor has to work in
    spite of those, not because of them.

    Returns a list of dicts with the signal, the true shaft speed and a run id.
    """
    rng = np.random.default_rng(seed)
    runs = []
    for rid in range(n_runs):
        p = _run_params(rng, "normal")
        p["f_r"] = float(rng.uniform(*SWEEP_FR))
        x = _synth(rng, p, seconds, FS)
        runs.append({"x": x, "f_r": p["f_r"],
                     "run": rid, "params": p})
    return runs


def build_speed_table(runs, win=WIN, hop=HOP, fs=FS):
    """(X, y, groups) for the soft sensor.  y is the true shaft speed in Hz."""
    X, y, g = [], [], []
    for r in runs:
        w = frame(r["x"], win, hop)
        X.append(features_of(w, fs))
        y += [r["f_r"]] * w.shape[0]
        g += [r["run"]] * w.shape[0]
    return np.vstack(X), np.array(y), np.array(g)


# =========================================== Lecture 10: the fleet campaign
# A year of monitoring across a fleet of nominally identical rigs.  Faults are
# RARE, and the ones you catch early are mild -- which is the whole point of
# catching them early, and the reason recall is hard.
FLEET_N = {"normal": 132, "imbalance": 12, "bearing": 6}


def _fleet_params(rng, state):
    """Run parameters for the fleet.  Fault severity spans incipient to severe."""
    p = _run_params(rng, state)
    if state == "imbalance":
        # a light trial mass barely shifts 1x; a thrown blade is unmistakable
        p["a_1x"] = float(rng.uniform(0.42, 1.10))
    if state == "bearing":
        # an early spall rings faintly; a spalled race is loud
        p["a_imp"] = float(rng.uniform(0.22, 1.40))
    return p


def make_fleet(seed=23, counts=None, seconds=RUN_SECONDS, fs=FS):
    """One acquisition per machine per month, across a fleet.

    Returns runs in a shuffled order with a 'label' and a 'severity' field.
    Prevalence is industrial, not academic: most machines are fine, a few have
    imbalance, and bearing spalls are rarer still.
    """
    counts = counts or FLEET_N
    rng = np.random.default_rng(seed)
    runs, rid = [], 0
    for state, n in counts.items():
        for _ in range(n):
            p = _fleet_params(rng, state)
            x = _synth(rng, p, seconds, fs)
            sev = (p["a_1x"] if state == "imbalance"
                   else p["a_imp"] if state == "bearing" else 0.0)
            runs.append({"x": x, "label": state, "run": rid,
                         "severity": float(sev), "params": p})
            rid += 1
    order = rng.permutation(len(runs))
    return [runs[i] for i in order]


# ============================================ Lecture 10: the drift campaign
# The same healthy machine, measured monthly for four years.  Nothing breaks --
# but the sensor mount relaxes, the ambient noise floor rises and the frame
# resonance walks.  A threshold set in the first year will not survive.
def make_drift_campaign(seed=31, n_months=48, fault_from=42, seconds=RUN_SECONDS,
                        fs=FS):
    """Healthy runs in time order with a slow baseline drift, then a real fault.

    Each run carries 'month'.  Runs from `fault_from` onward are a genuine
    bearing spall, so the last months contain something a detector SHOULD fire
    on -- everything before that is a healthy machine that merely looks
    different from how it looked in month 0.
    """
    rng = np.random.default_rng(seed)
    runs = []
    for m in range(n_months):
        t = m / (n_months - 1)                       # 0 -> 1 over the campaign
        faulty = m >= fault_from
        p = _run_params(rng, "bearing" if faulty else "normal")
        p["noise"] = float(0.06 + 0.050 * t + rng.normal(0, 0.005))
        p["a_struct"] = float(0.04 + 0.045 * t + rng.normal(0, 0.004))
        p["f_struct"] = float(180.0 + 110.0 * t + rng.normal(0, 8.0))
        p["gain"] = float(1.0 + 0.10 * t + rng.normal(0, 0.015))
        if faulty:
            p["a_imp"] = 1.10
        x = _synth(rng, p, seconds, fs)
        runs.append({"x": x, "label": "bearing" if faulty else "normal",
                     "run": m, "month": m, "params": p})
    return runs

In [9]:
def all_features_py(buf):
    '''The twelve features, device-style: plain Python, no NumPy.'''
    t = time_features_py(buf)
    f = freq_features_py(buf, t[0])
    return (t[0], t[1], t[2], t[3], t[4], t[5], t[6],
            f[0], f[1], f[2], f[3], f[4])

Now the comparison, on a bearing run neither implementation has seen.

In [10]:
import rig

rng = np.random.default_rng(41)
p = rig._run_params(rng, "bearing")
p["a_imp"] = 0.9
x = rig._synth(rng, p, 2.0, FS)

# quantise to int16 exactly as the ADC would, so both paths see identical numbers
scale = float(np.abs(x).max()) / 32767.0
xq = np.round(x / scale).astype(np.int16).astype(np.float64) * scale

W = frame(xq, 256, 128)
ref = features_of(W)                                  # NumPy, float64
dev = np.array([all_features_py([float(v) for v in row]) for row in W])

err = np.abs(dev - ref).max()
print(f"windows compared : {len(W)}")
print(f"max abs error    : {err:.2e}")
print("verdict          :", "PASS" if err < 1e-6 else "FAIL -- do not deploy")

windows compared : 30
max abs error    : 3.00e-08
verdict          : PASS


Eight significant figures of agreement, from two implementations that share no code.
That is what "verified" should mean.

## 3. What it costs

Now measure. The honest measurement happens on the ESP32 in Wokwi — `bench.py` in the
lab package does exactly this — but we can measure the same code here and learn the
part that transfers.

Timings measured for this lecture under **MicroPython 1.22.1** on a desktop
build (median of 7 runs of 60 repetitions each):

| stage | µs per window | as a share of the 128 ms window |
|---|---|---|
| 7 time-domain features | 113 | 0.09 % |
| 256-point FFT | 957 | 0.75 % |
| 5 frequency features (incl. FFT) | 1036 | 0.81 % |
| all 12 features | 1147 | 0.90 % |
| classifier (36 MACs) | 6 | 0.005 % |
| alarm (12 z-scores) | 3 | 0.002 % |
| **total** | **1156** | **0.90 %** |

The absolute numbers are useless to you: a desktop MicroPython build is tens of times
faster than an ESP32 at 240 MHz. The **ratios** are what transfer, because they are
ratios of interpreter operations and the interpreter is the same one.

In [11]:
RATIO_FFT = 8.5        # FFT / time-domain features, measured
RATIO_FULL = 10.2       # whole pipeline / time-domain features
RATIO_LITE = 3.2        # lite pipeline / time-domain features
BUDGET_MS = 128.0

print(f"the FFT costs {RATIO_FFT} x the seven time-domain features")
print(f"the whole pipeline costs {RATIO_FULL} x them")
print()
print("DESIGN RULE")
print(f"  if your Lab 8 measurement of the time-domain features is under "
      f"{BUDGET_MS / RATIO_FULL:.1f} ms,")
print(f"  the full twelve-feature pipeline fits the {BUDGET_MS:.0f} ms window.")
print(f"  if it is between {BUDGET_MS / RATIO_FULL:.1f} and "
      f"{BUDGET_MS / RATIO_LITE:.1f} ms, use the lite path of section 4.")

the FFT costs 8.5 x the seven time-domain features
the whole pipeline costs 10.2 x them

DESIGN RULE
  if your Lab 8 measurement of the time-domain features is under 12.5 ms,
  the full twelve-feature pipeline fits the 128 ms window.
  if it is between 12.5 and 40.0 ms, use the lite path of section 4.


That is a rule you can apply to a board this notebook has never run on, and it is the
reason we measured ratios rather than quoting somebody's benchmark.

You can time the same functions here, under CPython, as a sanity check on the shape of
the profile. Expect CPython to be perhaps 30–60× faster than MicroPython on an ESP32
and about 5–10× faster than the desktop MicroPython numbers above — but expect the FFT
to still dominate by roughly the same factor.

In [12]:
buf = [float(v) for v in W[0]]
REPS = 200


def timeit(fn, reps=REPS):
    fn()
    t0 = time.perf_counter()
    for _ in range(reps):
        fn()
    return 1e6 * (time.perf_counter() - t0) / reps


tf = time_features_py(buf)
t_time = timeit(lambda: time_features_py(buf))
t_fft = timeit(lambda: _fft(buf, tf[0]))
t_all = timeit(lambda: all_features_py(buf))

print(f"CPython, this machine")
print(f"  7 time-domain features {t_time:8.1f} us")
print(f"  256-point FFT          {t_fft:8.1f} us   ({t_fft / t_time:.1f} x)")
print(f"  all 12 features        {t_all:8.1f} us   ({t_all / t_time:.1f} x)")
print()
print(f"  measured under MicroPython for the lecture: FFT was "
      f"{RATIO_FFT:.1f} x, all 12 were {RATIO_FULL:.1f} x")

CPython, this machine
  7 time-domain features     34.8 us
  256-point FFT             530.5 us   (15.3 x)
  all 12 features           582.0 us   (16.7 x)

  measured under MicroPython for the lecture: FFT was 8.5 x, all 12 were 10.2 x


The ratios are not identical to the MicroPython ones — expect somewhere in the 10–20×
range here against about 10× there — because the two interpreters price `array('f')`
indexing, float boxing and function calls differently, and those are exactly what the
two loops do in different proportions.

What survives the change of machine is the *ordering* and the *order of magnitude*: the
FFT is an O(N log N) inner loop written in interpreted Python and it dominates anywhere.
That is a property of the algorithm, not of the clock. It is also the reason the design
rule above is stated as a ratio you measure on your own board rather than a microsecond
figure you inherit from a lecture.

> **Measure on the target.** Everything above is a way of predicting what you will
> measure, not a substitute for measuring it. Lab 11 asks you to run `bench.py` on the
> ESP32 in Wokwi, and the number it prints is the only one you may quote in a report.

## 4. Does the FFT earn its place?

We now have the cost. The other half of the trade is the benefit, and we can measure
that too — by retraining the Lecture 9 model on three different feature sets and
comparing on shared splits.

The third set is the **fallback path**, `features_lite.py`, which computes nine
features with no spectrum at all:

* `e_1x_cheap` — the Goertzel algorithm evaluates three individual DFT bins (3, 4, 5 —
  the 20–42 Hz band) using a two-variable recurrence per bin. No array, no spectrum.
* `e_hi_cheap` — one band-pass biquad over 400–900 Hz, five multiplies per sample, and
  the mean square of its output.

Both are normalised by the window's AC power, so they are dimensionless band-energy
fractions like the originals.

In [13]:
"""NumPy mirrors of the device's cheap ('lite') feature path.

Used only to measure what the fallback would cost in accuracy.  The device
implementation is wokwi/features_lite.py and must agree with these.
"""
import numpy as np
from rig import FS, WIN, HOP, frame, time_features

DF = FS / WIN
E1X_BINS = (3, 4, 5)                     # 23.4, 31.2, 39.1 Hz -> the 20-42 band
HI_LO, HI_HI = 400.0, 900.0


def goertzel_power(w, k, n=WIN):
    """Power in FFT bin k of each row of w, rectangular window."""
    coeff = 2.0 * np.cos(2.0 * np.pi * k / n)
    s1 = np.zeros(len(w))
    s2 = np.zeros(len(w))
    ac = w - w.mean(1, keepdims=True)
    for i in range(n):
        s0 = ac[:, i] + coeff * s1 - s2
        s2, s1 = s1, s0
    return s1 * s1 + s2 * s2 - coeff * s1 * s2


def biquad_bandpass(lo=HI_LO, hi=HI_HI, fs=FS):
    """One RBJ constant-skirt band-pass biquad, returned as (b, a)."""
    f0 = np.sqrt(lo * hi)
    bw = hi / lo
    w0 = 2.0 * np.pi * f0 / fs
    alpha = np.sin(w0) * np.sinh(np.log(2.0) / 2.0 * np.log2(bw) * w0 / np.sin(w0))
    b = np.array([alpha, 0.0, -alpha])
    a = np.array([1.0 + alpha, -2.0 * np.cos(w0), 1.0 - alpha])
    return b / a[0], a / a[0]


B_HI, A_HI = biquad_bandpass()


def band_power(w, b=B_HI, a=A_HI):
    """Mean square of the band-passed signal, one value per row."""
    ac = w - w.mean(1, keepdims=True)
    y = np.zeros_like(ac)
    x1 = x2 = y1 = y2 = np.zeros(len(w))
    for i in range(ac.shape[1]):
        x0 = ac[:, i]
        y0 = b[0] * x0 + b[1] * x1 + b[2] * x2 - a[1] * y1 - a[2] * y2
        y[:, i] = y0
        x2, x1 = x1, x0
        y2, y1 = y1, y0
    return (y ** 2).mean(1)


LITE_NAMES = ["mean", "rms", "std", "ptp", "crest", "kurt", "zcr",
              "e_1x_cheap", "e_hi_cheap"]


def lite_features(w):
    """Seven time-domain features plus two cheap spectral ones."""
    t = time_features(w)
    ac = w - w.mean(1, keepdims=True)
    total = (ac ** 2).sum(1)
    total = np.maximum(total, 1e-20)
    e1 = sum(goertzel_power(w, k) for k in E1X_BINS) / total
    ehi = band_power(w) * w.shape[1] / total
    return np.column_stack([t, e1, ehi])


def build_lite_table(runs, win=WIN, hop=HOP):
    X, y, g = [], [], []
    for r in runs:
        w = frame(r["x"], win, hop)
        X.append(lite_features(w))
        y += [r["label"]] * w.shape[0]
        g += [r["run"]] * w.shape[0]
    return np.vstack(X), np.array(y), np.array(g)

In [14]:
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler

runs = make_dataset(seed=7)
Xf, y, g = build_table(runs)                 # 12 features
Xl, yl, gl = build_lite_table(runs)          # 9 lite features
TIME_IDX = [FEATURE_NAMES.index(k) for k in
            ("mean", "rms", "std", "ptp", "crest", "kurt", "zcr")]
G = np.unique(g)


def split3(seed):
    rng = np.random.default_rng(seed)
    p = rng.permutation(G)
    a, b = int(0.6 * len(G)), int(0.8 * len(G))
    return np.isin(g, p[:a]), np.isin(g, p[a:b]), np.isin(g, p[b:])


def score_set(A, seeds=8):
    '''Tune C on validation, refit on train+val, report test -- as in Lecture 9.'''
    out = []
    for s in range(seeds):
        a, b, c = split3(s)
        sc = StandardScaler().fit(A[a])
        Z = sc.transform(A)
        best, bh = -1.0, None
        for Cv in np.logspace(-4, 2, 13):
            v = LinearSVC(C=Cv, max_iter=20000).fit(Z[a], y[a]).score(Z[b], y[b])
            if v > best:
                best, bh = v, Cv
        out.append(LinearSVC(C=bh, max_iter=20000)
                   .fit(Z[a | b], y[a | b]).score(Z[c], y[c]))
    return np.array(out)


acc_full = score_set(Xf)
acc_time = score_set(Xf[:, TIME_IDX])
acc_lite = score_set(Xl)

print(f"{'feature set':<24}{'accuracy':>10}{'vs full (paired)':>22}")
print("-" * 56)
print(f"{'12 features (full FFT)':<24}{acc_full.mean():>10.3f}{'baseline':>22}")
for nm, a in (("7 time-domain only", acc_time), ("9 lite (no FFT)", acc_lite)):
    d = a - acc_full
    print(f"{nm:<24}{a.mean():>10.3f}"
          f"{f'{d.mean():+.3f} ± {d.std() / np.sqrt(len(d)):.3f}':>22}")

feature set               accuracy      vs full (paired)
--------------------------------------------------------
12 features (full FFT)       0.880              baseline
7 time-domain only           0.869        -0.011 ± 0.009
9 lite (no FFT)              0.881        +0.001 ± 0.006


Read that table slowly, because it does not say what a lecture on FFTs wants it to say.

The FFT buys **+0.011 ± 0.009** accuracy
over the time-domain features alone, and costs 8× their compute. And
the two cheap substitutes recover essentially all of it —
+0.001 ± 0.006, a difference smaller than
its own standard error — for about a third of the cost.

So why does this course deploy the full FFT?

**Because the model on the device must be the model you validated.** The Lecture 9
classifier was trained, tuned and tested on those twelve features; the Lecture 10
baseline was fitted on the same twelve. Deploying the lite path means retraining both,
re-validating both, re-exporting both, and re-doing Lecture 10's threshold study on the
new features. That is a real cost, and it buys you compute you have just measured and
do not need: the pipeline uses
0.9 % of the window on this host and
there is no crisis to solve.

Use the lite path when `bench.py` on **your** board says the full one does not fit.
Then retrain, re-validate, and say in the report that you did.

> The general point outlives the specific numbers. **A feature that is expensive and
> uninformative is an easy decision. The hard case is a feature that is expensive and
> mildly informative** — and you cannot tell which case you are in without measuring
> both halves.

In [15]:
def cheap_freq(buf, mean, s2):
    """(e_1x_cheap, e_hi_cheap) without ever forming a spectrum."""
    if s2 <= 0.0:
        return 0.0, 0.0

    # --- Goertzel, three bins at once
    p = 0.0
    for c in _GO_COEF:
        s1 = 0.0
        s2g = 0.0
        for v in buf:
            s0 = (v - mean) + c * s1 - s2g
            s2g = s1
            s1 = s0
        p += s1 * s1 + s2g * s2g - c * s1 * s2g

    # --- one band-pass biquad, direct form I
    x1 = 0.0
    x2 = 0.0
    y1 = 0.0
    y2 = 0.0
    acc = 0.0
    for v in buf:
        x0 = v - mean
        y0 = B0 * x0 + B2 * x2 - A1 * y1 - A2 * y2
        acc += y0 * y0
        x2 = x1
        x1 = x0
        y2 = y1
        y1 = y0

    return p / s2, acc / s2

Twelve lines and no array. That is the shape of most good embedded substitutions: a
recurrence that carries two state variables instead of a buffer that carries 256.

## 5. Exporting the classifier

The Lecture 9 model is a `LinearSVC` inside a `StandardScaler` pipeline. On the device
that is 63 numbers and, if you transcribe it naively, 12 subtractions and 12 divisions
before every inference.

It does not have to be. A linear model on standardised features *is* a linear model on
raw features:

$$
\mathbf{w}\cdot\frac{\mathbf{x}-\boldsymbol{\mu}}{\boldsymbol{\sigma}} + b
\;=\;
\underbrace{\left(\frac{\mathbf{w}}{\boldsymbol{\sigma}}\right)}_{\text{new weights}}\cdot\;\mathbf{x}
\;+\;
\underbrace{\left(b - \mathbf{w}\cdot\frac{\boldsymbol{\mu}}{\boldsymbol{\sigma}}\right)}_{\text{new bias}}
$$

Fold the scaler in and the standardiser disappears completely — not "is optimised", but
*does not exist* on the device.

In [16]:
clf = json.load(open("lecture9_model.json"))
mu = np.array(clf["scaler_mean"])
sd = np.array(clf["scaler_scale"])
Wc = np.array(clf["coef"])
bc = np.array(clf["intercept"])

W_fold = Wc / sd
b_fold = bc - (Wc * (mu / sd)).sum(axis=1)

# prove the fold on random feature vectors drawn from the right scale
probe = np.random.default_rng(0).normal(0, 1, (500, len(mu))) * sd + mu
orig = ((probe - mu) / sd) @ Wc.T + bc
fold = probe @ W_fold.T + b_fold
print(f"max |difference| over 500 probes: {np.abs(orig - fold).max():.2e}")

before = Wc.size + bc.size + 2 * len(mu)
after = W_fold.size + b_fold.size
print(f"\nnumbers in flash : {before} -> {after}")
print(f"divides/inference: {len(mu)} -> 0")
print(f"subtracts/infer. : {len(mu)} -> 0")
print(f"what remains     : {Wc.size} multiply-accumulates and an argmax over "
      f"{len(clf['classes'])}")

max |difference| over 500 probes: 2.22e-15

numbers in flash : 63 -> 39
divides/inference: 12 -> 0
subtracts/infer. : 12 -> 0
what remains     : 36 multiply-accumulates and an argmax over 3


Exact to the last bit of double precision, because it is algebra, not approximation.

Now generate the device file. Not by hand — by script, so that retraining the model
regenerates `model.py` and nobody ever transcribes a coefficient by eye.

In [17]:
def fmt(vals, per_line=4, indent=8):
    out, line = [], []
    for v in vals:
        line.append(f"{v:.7g}")
        if len(line) == per_line:
            out.append(" " * indent + ", ".join(line) + ",")
            line = []
    if line:
        out.append(" " * indent + ", ".join(line) + ",")
    return "\n".join(out)


src = "CLASSES = {!r}\n\nCOEF = (\n{}\n)\n\nINTERCEPT = (\n{}\n)\n".format(
    tuple(clf["classes"]),
    "\n".join("    (\n" + fmt(r) + "\n    )," for r in W_fold),
    fmt(b_fold, per_line=3, indent=4))

print(src[:420] + "...\n")
print(f"generated source: {len(src)} characters")

CLASSES = ('bearing', 'imbalance', 'normal')

COEF = (
    (
        3.635836, -0.2896101, -0.2896101, 0.1041714,
        0.7507597, 0.3303283, -2.457207, -0.002660429,
        -0.6664433, -3.94146, -15.8365, 3.429931,
    ),
    (
        0.134412, 1.955565, 1.955565, 0.4972783,
        -0.2524321, -0.1308921, -1.382013, -3.325341e-05,
        1.335247, 3.491781, -1.134162, -0.9265764,
    ),
    (
        -3.385322...

generated source: 623 characters


### What exports and what does not

| model | exports to | why |
|---|---|---|
| linear / logistic | tens of numbers | prediction is one dot product |
| decision tree | a nest of `if` statements | prediction is a path, no arithmetic |
| small ensemble (< 20 shallow trees) | a few KB | same, repeated, then a vote |
| **k-NN** | **the entire training set** | the training set *is* the model |
| **random forest, untuned** | **megabytes** | hundreds of trees × hundreds of nodes |
| **anything that allocates while predicting** | **do not** | heap fragmentation, eventual failure |

Lecture 9 chose the linear SVM on a joint argument about accuracy and size, and this
table is the second half of that argument arriving. A tool such as **m2cgen** will
generate C, Python or a dozen other languages from a fitted scikit-learn model and is
worth knowing about — but for a linear model the generated code is the few dozen lines
above, and generating it yourself means you know exactly what is in flash.

## 6. Exporting the alarm

The Lecture 10 alarm is even smaller: a mean and a standard deviation per feature, one
threshold, and a persistence rule. Twenty-five numbers, no model at all.

Two device-side changes are worth making.

In [18]:
alm = json.load(open("lecture10_alarm.json"))
amu = np.array(alm["scaler_mean"])
asd = np.array(alm["scaler_scale"])
inv = 1.0 / asd                            # store the reciprocal, multiply on device

print(f"features      : {len(amu)}")
print(f"numbers stored: {2 * len(amu) + 1}")
print(f"threshold     : {alm['threshold']:.3f}  "
      f"({alm['threshold_quantile']:.0%} quantile of healthy scores)")
print(f"persistence   : {alm['persistence']['m']} of the last "
      f"{alm['persistence']['n']} windows")
print(f"review after  : {alm['review_after_months']} months")

features      : 12
numbers stored: 25
threshold     : 4.038  (99% quantile of healthy scores)
persistence   : 3 of the last 5 windows
review after  : 12 months


1. **Store the reciprocal of each standard deviation.** The z-score becomes
   `(x - mean) * inv_sd`, a multiply instead of a divide. On the ESP32 a float divide
   costs several times a float multiply, and we do twelve of them per window forever.
2. **Return which feature fired, not only how large the score was.** An alarm that says
   "4.7" tells a technician nothing; an alarm that says "`e_bpfo`, over threshold" sends
   them to the bearing.

The second point carries a caveat the device code states explicitly, and section 8 will
show why it matters.

In [19]:
FEATURES = tuple(alm["feature_names"])
MEAN = tuple(amu)
INV_SD = tuple(inv)
THRESHOLD = alm["threshold"]
PERSIST_M, PERSIST_N = alm["persistence"]["m"], alm["persistence"]["n"]
_hist = [0] * PERSIST_N
_i = 0


def worst_feature(x):
    '''(score, name) -- the largest absolute z-score and which feature it was.'''
    worst, at = 0.0, 0
    for j in range(len(MEAN)):
        z = (x[j] - MEAN[j]) * INV_SD[j]
        if z < 0.0:
            z = -z
        if z > worst:
            worst, at = z, j
    return worst, FEATURES[at]


def update(x):
    '''Feed one window. Returns (raised, score) after the persistence rule.'''
    global _i
    s, _ = worst_feature(x)
    _hist[_i] = 1 if s > THRESHOLD else 0
    _i = (_i + 1) % PERSIST_N
    return (sum(_hist) >= PERSIST_M), s


def reset():
    global _i
    for k in range(PERSIST_N):
        _hist[k] = 0
    _i = 0


print("the whole alarm, in", len(MEAN) * 2 + 1, "numbers and three short functions")

the whole alarm, in 25 numbers and three short functions


The ring buffer is the persistence rule from Lecture 10, and it is three lines because
a fixed-length list with a rotating index allocates nothing. A `collections.deque` would
be more elegant and would import a module the device does not need.

## 7. What single precision costs

The ESP32 has single-precision floating point in hardware and double precision in
software, which is to say it does not really have double precision. MicroPython on the
ESP32 uses 32-bit floats. Everything in this course so far has been float64.

That is a change of about seven decimal digits of precision. Does it matter?

In [20]:
f64 = features_of(W)
f32 = features_of(W.astype(np.float32)).astype(np.float64)
rel = np.abs(f32 - f64) / np.maximum(np.abs(f64), 1e-12)

tbl = pd.DataFrame({"max relative error": rel.max(0)}, index=FEATURE_NAMES)
print(tbl.sort_values("max relative error", ascending=False)
        .to_string(float_format=lambda v: f"{v:.2e}"))
print(f"\nworst feature: {FEATURE_NAMES[int(rel.max(0).argmax())]} at "
      f"{rel.max():.1e} relative -- about {rel.max() * 100:.1e} %")

          max relative error
kurt                3.41e-07
e_bpfo              1.82e-07
crest               1.39e-07
mean                9.83e-08
e_2x                8.89e-08
rms                 8.80e-08
std                 8.80e-08
ptp                 5.76e-08
e_hi                5.21e-08
e_1x                4.29e-08
dom_freq            0.00e+00
zcr                 0.00e+00

worst feature: kurt at 3.4e-07 relative -- about 3.4e-05 %


The worst feature is **kurt** at 3e-07 relative error,
which is roughly 3e-05 %. Kurtosis is worst because it is a
fourth moment — errors in the deviations get raised to the fourth power along with the
deviations.

Put that beside the numbers it feeds. The classifier's smallest healthy margin in
section 8 is about 1.5; the alarm's threshold is
4.0 on a z-score scale. A perturbation in the seventh decimal
place cannot move either decision. **Single precision is fine here, and we checked
rather than assumed.**

It would not always be fine. If you accumulate a sum over a million samples instead of
256, or subtract two nearly equal large numbers, float32 will bite. The discipline is
the same one as everywhere else in this course: state the tolerance your decision needs,
then measure the error you actually have.

This is also why Lab 8 asked for agreement to three decimal places and not to eight.

## 8. Two models, and a fault neither was trained on

The device runs the classifier and the alarm together. Students reasonably ask why —
the classifier already outputs `normal`, `imbalance` or `bearing`, so what is the alarm
for?

Here is the answer, as an experiment. We synthesise a fourth machine state: a **loose
mounting bolt**, which puts a large structural resonance at 330 Hz where no training run
had one. It appears in no training set in this course. The classifier has three output
labels and no way to say "none of these".

In [21]:
EXTRA = {"unknown": dict(a_struct=0.45, f_struct=330.0)}


def device_predict(x):
    '''The Lecture 9 classifier, device-style: 36 MACs and an argmax.'''
    s = [b_fold[c] + float(np.dot(W_fold[c], x)) for c in range(len(b_fold))]
    best = int(np.argmax(s))
    second = max(v for i, v in enumerate(s) if i != best)
    return clf["classes"][best], s[best] - second


rows = []
rng = np.random.default_rng(41)
for state in list(CLASSES) + list(EXTRA):
    p = rig._run_params(rng, "normal" if state in EXTRA else state)
    if state == "bearing":
        p["a_imp"] = 0.9
    p.update(EXTRA.get(state, {}))
    xs = rig._synth(rng, p, 2.0, FS)
    sc = float(np.abs(xs).max()) / 32767.0
    xs = np.round(xs / sc).astype(np.int16).astype(np.float64) * sc
    F = features_of(frame(xs, 256, 128))

    reset()
    lab, mar, sco, drv, rai = [], [], [], [], []
    for row in F:
        v = [float(t) for t in row]
        l, m = device_predict(v)
        s, d = worst_feature(v)
        r, _ = update(v)
        lab.append(l); mar.append(m); sco.append(s); drv.append(d); rai.append(r)
    rows.append(dict(state=state,
                     called=max(set(lab), key=lab.count),
                     margin=float(np.median(mar)),
                     score=float(np.median(sco)),
                     driver=max(set(drv), key=drv.count),
                     raised=float(np.mean(rai))))

print(pd.DataFrame(rows).to_string(
    index=False, float_format=lambda v: f"{v:.3f}"))

    state    called  margin    score   driver  raised
   normal    normal   1.470    1.610     kurt   0.000
imbalance imbalance   1.175    4.533      rms   0.933
  bearing   bearing   2.342    4.280     kurt   0.433
  unknown    normal   7.468 1289.393 dom_freq   0.933


Read the last row twice.

The classifier calls the loose mounting bolt **normal**, with a margin of about
7.5 — more than
5 times
its margin on an actually healthy machine
(1.5). It is not merely wrong; it is *more
confident about a broken machine than about a healthy one*.

This is not a bug in the model. A linear SVM's decision value is the distance from a
separating hyperplane, and a point far from every training example on the `normal` side
gets a large distance. The model was asked "which of these three?" and it answered
correctly: of the three, this is most like `normal`. Nobody told it there was a fourth
option, and the margin is not a probability of being right.

The alarm, meanwhile, scores it at about **1289**
against a threshold of 4.0, and raises on
93 % of windows. It has never seen this
fault either — but it was never asked to recognise faults, only to recognise *healthy*,
and this is not that.

### The caveat on the number

The driver is `dom_freq`, and the score is enormous. Do not read that as severity. The
healthy standard deviation of `dom_freq` is a fraction of a hertz, because a healthy
machine's spectrum peaks in the same bin every single time. A shift of a few bins is
therefore hundreds of standard deviations, and a shift of a few bins is a small physical
change.

**Report the name; treat the number as "over threshold" and nothing more.** A z-score
on a near-constant feature is a detector, not a scale. This is exactly why
`worst_feature()` returns a name at all.

## 9. Fail-safe: who owns the actuator

Section 8 settles a design question. When the classifier and the alarm disagree, which
one acts?

**The alarm.** Not because it is cleverer — it cannot even name the fault — but because
its false-alarm rate was measured on healthy data in Lecture 10, and the classifier's
confidence on an unfamiliar input was never measured and cannot be. You are allowed to
act on a quantity you have characterised.

So on the device: the alarm owns the actuator, and the classifier annotates the alarm.

```python
label, margin = classify(x)          # what to write in the log
raised, score = alarm.update(x)      # what to do about it
if raised != raised_last:
    set_actuator(raised)             # only the alarm reaches the outside world
```

And a device that is deciding by itself must behave when parts of it fail.

| what fails | what the device does |
|---|---|
| a feature raises (bad sample, divide by zero) | hold the last actuator state, log the window, do not guess |
| the model raises | the alarm still runs; the label becomes `"model-error"` |
| the classifier is barely deciding (`margin < 0.25`) | the label becomes `"uncertain"`; the alarm is unaffected |
| the network is down | nothing changes — no decision in this loop needs it |
| the actuator write fails | swallow the exception, keep the loop alive, log it |

The last row of the table is the whole argument for edge AI in one line. Every decision
on this device is made from twelve numbers the device computed itself. The network
carries the *report*, not the *decision*, and Lecture 12 is about that report.

In [22]:
MIN_MARGIN = 0.25


def classify(x):
    '''Name the fault, or admit you cannot. Never raises.'''
    try:
        label, margin = device_predict(x)
        if margin < MIN_MARGIN:
            return "uncertain", margin
        return label, margin
    except Exception:
        return "model-error", 0.0


def one_window(buf, raised_last):
    '''The device loop body, with its fail-safes, in eleven lines.'''
    try:
        x = all_features_py(buf)
    except Exception:
        return raised_last, "feature-error", 0.0, "-"      # hold, do not guess
    label, margin = classify(x)
    raised, score = update(x)
    _, driver = worst_feature(x)
    return raised, label, score, driver


reset()
print("the bearing run of section 2, one window at a time")
print(f"{'win':>4}{'label':>14}{'score':>10}{'driver':>12}{'raised':>8}")
state = False
for i, row in enumerate(W[:8]):                    # W is 256 raw samples per window
    state, lab, sc, dr = one_window([float(v) for v in row], state)
    print(f"{i:>4}{lab:>14}{sc:>10.2f}{dr:>12}{str(state):>8}")

print()
print("and the same loop when the sensor hands it nonsense")
bad = [0.0] * 4                                    # a truncated window
state, lab, sc, dr = one_window(bad, state)
print(f"{'--':>4}{lab:>14}{sc:>10.2f}{dr:>12}{str(state):>8}   <- last state held")

the bearing run of section 2, one window at a time
 win         label     score      driver  raised
   0       bearing      4.70        kurt   False
   1       bearing      6.30        kurt   False
   2       bearing      6.17        kurt    True
   3       bearing      3.88        kurt    True
   4       bearing      3.96        kurt    True
   5       bearing      3.87       crest   False
   6       bearing      4.41        kurt   False
   7       bearing      6.40        kurt   False

and the same loop when the sensor hands it nonsense
  -- feature-error      0.00           -   False   <- last state held


Note what `one_window` returns when feature extraction fails: the *previous* actuator
state, unchanged. A device that drops to "safe" on every glitch will trip the line on
electrical noise; a device that drops to "no alarm" on every glitch is a device that
fails silently. Holding the last state and logging the glitch is the compromise that
survives a real plant — and the log is what tells you the glitches are becoming
frequent.

## 10. Does it fit?

The last check. Lecture 9 set a budget of 200 KB of RAM for
everything on the device, and we have been spending flash and heap all lecture without
counting.

In [23]:
sizes = {
    "alarm.py": 2608,
    "bench.py": 2878,
    "features.py": 4923,
    "features_lite.py": 3293,
    "main.py": 3838,
    "model.py": 1764,
    "wave_bearing.py": 12112,
    "wave_imbalance.py": 12116,
    "wave_normal.py": 12111,
    "wave_unknown.py": 12112
}
HEAP_PER_PASS = 992
RAM_BUDGET = 200 * 1024

code_files = ("features.py", "features_lite.py", "model.py", "alarm.py", "main.py")
code_bytes = sum(sizes[k] for k in code_files)

print("deployed MicroPython source")
for k in code_files:
    print(f"  {k:<22}{sizes[k]:>9,} B")
print(f"  {'total':<22}{code_bytes:>9,} B")
print()
print(f"  {'the two models alone':<22}{sizes['model.py'] + sizes['alarm.py']:>9,} B")
print(f"  {'heap per inference':<22}{HEAP_PER_PASS:>9,} B")
print(f"  {'RAM budget, Lecture 9':<22}{RAM_BUDGET:>9,} B")
print(f"  {'headroom':<22}{RAM_BUDGET - HEAP_PER_PASS - code_bytes:>9,} B")

deployed MicroPython source
  features.py               4,923 B
  features_lite.py          3,293 B
  model.py                  1,764 B
  alarm.py                  2,608 B
  main.py                   3,838 B
  total                    16,426 B

  the two models alone      4,372 B
  heap per inference          992 B
  RAM budget, Lecture 9   204,800 B
  headroom                187,382 B


16,426 bytes of source for the entire edge pipeline — features, FFT,
fallback, both models and the main loop — and 992 bytes of heap per
inference, against a 200 KB budget.

The heap number is the one to dwell on. It is small and, more importantly, it is
*constant*: the FFT buffers were allocated once at import, so the loop churns no memory
and the garbage collector has almost nothing to do. A version that allocated its buffers
per window would show a few kilobytes here, run fine on the bench, and die after a week
of fragmentation.

Free-running embedded code is judged on allocation per iteration, not on peak memory.

## The protocol

Deploying a model to a microcontroller, in the order the steps must happen:

1. **Write the feature code in plain Python and verify it against NumPy** on real data,
   to at least six significant figures, before timing anything.
2. **Measure the cost per stage on the target board.** `bench.py`, on the ESP32, not a
   desktop estimate.
3. **Compare the cost against the window duration**, not against your patience. The
   budget is set by the sensor.
4. **Test what each expensive stage buys** in accuracy, on shared splits, with a paired
   difference and its standard error.
5. **Fold the scaler into the coefficients** and assert the fold is exact.
6. **Generate the device file from the artefact by script.** Never transcribe a
   coefficient.
7. **Check single precision** against the smallest margin your decision depends on.
8. **Allocate every buffer once, at import.** Nothing in the loop may allocate.
9. **Decide who owns the actuator**, and write the fail-safe branch for each component
   that can fail.
10. **Run both models.** The classifier names what it knows; the alarm notices what
    nobody labelled.

## Exercises

1. **Measure your board.** Run `bench.py` on the ESP32 in Wokwi and fill in the table of
   section 3 with your own numbers. Does the full twelve-feature pipeline fit the 128 ms
   window? By what margin? Compare the FFT/time-features ratio you measure with the one
   quoted here and explain any difference — it is a property of the interpreter, not of
   the clock speed.

2. **Halve the window.** Change `WIN` to 128 and rebuild everything: the twiddle tables,
   the band-bin edges, the features, and the model. The FFT should get more than twice as
   cheap. Why more than twice? And what happens to the frequency resolution `DF`, and
   hence to the `e_1x` and `e_2x` bands, which are only 22 and 30 Hz wide?

3. **The lite path, properly.** Retrain the Lecture 9 classifier *and* re-fit the Lecture
   10 baseline on the nine lite features, re-export both, and run the section 8
   experiment again. Does the lite alarm still catch the loose mounting bolt? Which
   feature drives it now, and why?

4. **Fixed point.** Replace the classifier's float coefficients with 16-bit integers
   scaled by a common power of two, and do the dot product in integer arithmetic. Find
   the smallest scale factor that leaves every prediction on the test set unchanged.
   How many bits did you actually need, and what does that suggest about the precision
   the model carries?

5. **Goertzel against the FFT.** The lite path evaluates three bins. At how many bins
   does the Goertzel approach become more expensive than one 256-point FFT? Derive the
   crossover from operation counts, then measure it and explain the gap between the two
   answers.

6. **A worse unknown.** Design a fourth fault that the *alarm* also misses — one whose
   twelve features all sit inside the healthy range. (Hint: Lecture 10's PCA section is
   the clue.) What would you have to add to the device to catch it, and is it worth it?

7. **Budget under load.** `main.py` measures per-window time with `ticks_us`. Add a WiFi
   connection and an MQTT publish to the loop in Wokwi, and measure again. How much of
   the window does the network cost, and what does that do to the argument in section 1?

8. **Towards Lecture 12.** The device now produces, every 64 ms, a label, a margin, an
   anomaly score, a driver name and a raised flag. Publishing all of that at 15 Hz is
   absurd. Design the reporting policy: what is sent immediately, what is aggregated,
   what is only sent on change, and how many bytes per hour does your policy cost?

## References

**Edge inference and model export**

* Warden, P. and Situnayake, D. (2019). *TinyML: Machine Learning with TensorFlow Lite
  on Arduino and Ultra-Low-Power Microcontrollers.* O'Reilly. Chapters 1–3 for the
  architecture argument; the toolchain differs from ours, the reasoning does not.
* Banbury, C. et al. (2021). *MLPerf Tiny benchmark.* NeurIPS Datasets and Benchmarks.
  How the field measures latency and memory on microcontrollers.
* m2cgen — model to code generator, scikit-learn to plain source:
  https://github.com/BayesWitnesses/m2cgen

**Signal processing on small machines**

* Cooley, J. W. and Tukey, J. W. (1965). *An algorithm for the machine calculation of
  complex Fourier series.* Mathematics of Computation, 19(90), 297–301.
* Goertzel, G. (1958). *An algorithm for the evaluation of finite trigonometric series.*
  The American Mathematical Monthly, 65(1), 34–35.
* Bristow-Johnson, R. *Cookbook formulae for audio EQ biquad filter coefficients.* The
  source of the band-pass used in `features_lite.py`.
* Lyons, R. G. (2010). *Understanding Digital Signal Processing*, 3rd ed. Chapter 13 on
  efficient fixed-point and reduced-precision implementations.

**Numerics and reliability**

* Goldberg, D. (1991). *What every computer scientist should know about floating-point
  arithmetic.* ACM Computing Surveys, 23(1), 5–48. Section 7 of this notebook in full.
* Hoare, C. A. R. (1981). *The emperor's old clothes.* CACM, 24(2), 75–83. On what
  happens when a system is made fast before it is made correct.

**Tools**

* MicroPython documentation, *Maximising MicroPython speed*:
  https://docs.micropython.org/en/latest/reference/speed_python.html
* MicroPython `array` module: https://docs.micropython.org/en/latest/library/array.html
* Wokwi ESP32 simulator: https://wokwi.com/

Generated by Claude and customized by

<div align="center">
<img src="https://raw.githubusercontent.com/dewdotninja/sharing-github/refs/heads/master/dewninja_logo50.jpg" alt="dewninja"/>
</div>
<div align="center">dew.ninja 2026</div>